***00_模擬資料以便進行A & B***

**Dataset A：MSA（量測系統分析）**

- 10 片晶圓（Wafer）
- 3 位量測人員（Operator）
- 每人重複測量 2 次
- 使用同一台量測設備

同一台設備的量測結果，會不會受到人員與重複操作影響？

**Dataset B：Tool Matching（設備間匹配）**

- 30 片晶圓
- 3 台量測設備（Tool A/B/C）
- 每台設備重複測量 2 次
- 固定同一位操作人員

三台設備測量同一批晶圓時，結果是否存在系統性差異？

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)
ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
NOMINAL_CD = 45.0  # nm

# =====================================
# 1. Dataset A：MSA
# =====================================

msa_rows = []
wafers = [f"W{i:03d}" for i in range(1, 11)]
operators = ["OP01", "OP02", "OP03"]
repeats = [1, 2]

# 每片晶圓有自己的真實 CD
wafer_true_cd = {
    wafer: rng.normal(NOMINAL_CD, 0.8)
    for wafer in wafers
}

# 模擬不同操作人員的固定偏差
operator_bias = {
    "OP01": 0.00,
    "OP02": 0.08,
    "OP03": -0.06
}

# 模擬同一量測系統的重複量測誤差
repeat_sd = 0.12

for wafer in wafers:
    true_cd = wafer_true_cd[wafer]

    for operator in operators:
        for repeat in repeats:

            measurement_error = rng.normal(0, repeat_sd)

            measured_cd = (
                true_cd
                + operator_bias[operator]
                + measurement_error
            )

            msa_rows.append({
                "wafer_id": wafer,
                "lot_id": f"L{int(wafer[1:]) // 5 + 1:03d}",
                "tool_id": "Tool_A",
                "operator_id": operator,
                "repeat": repeat,
                "recipe": "CD_RECIPE_01",
                "true_cd_nm": true_cd,
                "measured_cd_nm": measured_cd,
                "timestamp": pd.Timestamp("2026-11-05")
            })

msa_df = pd.DataFrame(msa_rows)


# =====================================
# 2. Dataset B：Tool Matching
# =====================================

tool_rows = []

wafers = [f"W{i:03d}" for i in range(1, 31)]
tools = ["Tool_A", "Tool_B", "Tool_C"]

# 每台設備的系統性偏差
tool_bias = {
    "Tool_A": 0.00,
    "Tool_B": 0.25,
    "Tool_C": -0.18
}

# 同一台設備的隨機量測誤差
tool_repeat_sd = 0.10

for wafer in wafers:

    true_cd = rng.normal(NOMINAL_CD, 0.8)

    for tool in tools:
        for repeat in [1, 2]:

            measurement_error = rng.normal(
                0, tool_repeat_sd
            )

            measured_cd = (
                true_cd
                + tool_bias[tool]
                + measurement_error
            )

            tool_rows.append({
                "wafer_id": wafer,
                "lot_id": f"L{int(wafer[1:]) // 10 + 1:03d}",
                "tool_id": tool,
                "operator_id": "OP01",
                "repeat": repeat,
                "recipe": "CD_RECIPE_01",
                "true_cd_nm": true_cd,
                "measured_cd_nm": measured_cd,
                "timestamp": pd.Timestamp("2026-11-06")
            })

tool_df = pd.DataFrame(tool_rows)


# =====================================
# 3. 儲存資料
# =====================================

msa_df.to_csv(
    RAW_DIR / "msa_data.csv",
    index=False
)

tool_df.to_csv(
    RAW_DIR / "tool_matching_data.csv",
    index=False
)

print("資料產生完成！")
print("MSA dataset:", msa_df.shape)
print("Tool matching dataset:", tool_df.shape)

print("\nMSA preview:")
print(msa_df.head())

print("\nTool matching preview:")
print(tool_df.head())


資料產生完成！
MSA dataset: (60, 9)
Tool matching dataset: (180, 9)

MSA preview:
  wafer_id lot_id tool_id operator_id  repeat        recipe  true_cd_nm  \
0     W001   L001  Tool_A        OP01       1  CD_RECIPE_01   45.243774   
1     W001   L001  Tool_A        OP01       2  CD_RECIPE_01   45.243774   
2     W001   L001  Tool_A        OP02       1  CD_RECIPE_01   45.243774   
3     W001   L001  Tool_A        OP02       2  CD_RECIPE_01   45.243774   
4     W001   L001  Tool_A        OP03       1  CD_RECIPE_01   45.243774   

   measured_cd_nm  timestamp  
0       45.349301 2026-11-05  
1       45.337109 2026-11-05  
2       45.331697 2026-11-05  
3       45.459043 2026-11-05  
4       45.239875 2026-11-05  

Tool matching preview:
  wafer_id lot_id tool_id operator_id  repeat        recipe  true_cd_nm  \
0     W001   L001  Tool_A        OP01       1  CD_RECIPE_01   43.979451   
1     W001   L001  Tool_A        OP01       2  CD_RECIPE_01   43.979451   
2     W001   L001  Tool_B        OP01  